# Phase 3 Backtest Report

This notebook loads a pre-computed backtest JSON produced by
`bottlewatch-backtest` and renders the diagnostic charts used in the
Phase 3 executive summary.

Run the CLI first to generate the input:

```bash
bottlewatch-backtest --normalization-mode both --output data/reports/phase3_backtest.json
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPORT_PATH = Path("../data/reports/phase3_backtest.json")
report = json.loads(REPORT_PATH.read_text())
report.keys()

## 1. Basket cumulative returns

Equal-weight long and short basket returns over the backtest window.

In [ ]:
def basket_curve(baskets, side):
    """Return dates and cumulative returns for one basket side."""
    side_baskets = [b for b in baskets if b["side"] == side]
    side_baskets.sort(key=lambda b: b["eval_date"])
    dates = [b["eval_date"] for b in side_baskets]
    rets = [b["equal_weight_return"] for b in side_baskets if b["equal_weight_return"] is not None]
    cum = np.cumprod([1.0 + r for r in rets]) if rets else []
    return dates[: len(cum)], cum


fig, ax = plt.subplots(figsize=(10, 5))
for side, label in (("long", "Long"), ("short", "Short")):
    dates, cum = basket_curve(report["baskets"], side)
    if dates:
        ax.plot(dates, (cum - 1) * 100, label=label, marker="o")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Basket cumulative returns")
ax.set_xlabel("Evaluation date")
ax.set_ylabel("Cumulative return (%)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 2. IC distribution

Per-segment Spearman information coefficient with block-bootstrap 95% confidence
intervals and Benjamini-Hochberg false-discovery flags.

In [ ]:
rows = report["per_segment_ic"]
if rows:
    fig, ax = plt.subplots(figsize=(10, 5))
    segments = [r["segment"] for r in rows]
    rhos = [r["rho"] if r["rho"] is not None else 0.0 for r in rows]
    ci_lows = [r["ci_low"] if r["ci_low"] is not None else r["rho"] or 0.0 for r in rows]
    ci_highs = [r["ci_high"] if r["ci_high"] is not None else r["rho"] or 0.0 for r in rows]
    errs = [[max(0, r - cl), max(0, ch - r)] for r, cl, ch in zip(rhos, ci_lows, ci_highs)]
    colors = ["tab:green" if r["bh_rejected"] else "tab:blue" for r in rows]
    ax.barh(segments, rhos, color=colors, xerr=np.array(errs).T, capsize=4)
    ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Spearman rho")
    ax.set_title("Per-segment IC (green = BH rejected at FDR 10%)")
    plt.tight_layout()
    plt.show()
else:
    print("No per-segment IC rows available.")

## 3. Fixed vs rolling divergence

Mean absolute difference in B per segment and the count of regime flips
between fixed-band and rolling 5-year normalization.

In [ ]:
fvr = report.get("fixed_vs_rolling")
if fvr and fvr["per_segment"]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    segments = [r["segment"] for r in fvr["per_segment"]]
    diffs = [r["mean_abs_b_diff"] if r["mean_abs_b_diff"] is not None else 0.0 for r in fvr["per_segment"]]
    flips = [r["regime_flips"] for r in fvr["per_segment"]]

    ax1.barh(segments, diffs)
    ax1.set_xlabel("Mean |B_fixed - B_rolling|")
    ax1.set_title("Score divergence")

    ax2.barh(segments, flips)
    ax2.set_xlabel("Regime flips")
    ax2.set_title("Regime disagreement")

    plt.tight_layout()
    plt.show()
else:
    print("No fixed/rolling comparison available.")

## 4. Write executive summary

The markdown summary is written next to this notebook so it can be
checked into the repo as the Phase 3 research artifact.

In [ ]:
summary_path = Path("../docs/plans/2026-06-14-phase3-executive-summary.md")
overall_ic = report["overall_ic"]
overall_p = report["overall_p_value"]
n_segments = len(report["per_segment_ic"])
n_baskets = len(report["baskets"])
seed_warnings = len(report["seed_share_warning_dates"])

summary = f"""# Phase 3 Backtest Executive Summary

**Window:** {report['start']} to {report['end']}  
**Horizon:** {report['horizon']}  
**Forward return:** {report['forward_days']} days  
**Evaluation tuples:** {report['n_eval_points']} across {report['n_eval_dates']} dates  

## Headline results

- Overall IC: **{overall_ic:.3f if overall_ic is not None else 'n/a'}** (p={overall_p:.2e if overall_p is not None else 'n/a'})
- Per-segment IC rows: {n_segments}
- Basket rebalances: {n_baskets}
- Seed-share warning dates: {seed_warnings}

## Interpretation

The backtest is diagnostic, not a trading strategy. A positive overall IC
suggests the bottleneck score ranks forward returns better than chance;
per-segment CIs and BH-corrected p-values surface where that signal is
statistically robust. Large fixed-vs-rolling divergence or many regime
flips indicate the rolling 5-year band is materially changing the score
and should be monitored as the sub-score history matures.

## Data caveats

- Prices are the placeholder `data/processed/prices.csv`; real price data
  will change basket returns.
- Only `ingested_at` is used as the point-in-time proxy; late revisions
  and backfills can still leak future information.
- Fixed and rolling bands use the current band configuration and universe;
  historical recomputes are not fully point-in-time on calibration.
"""

summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(summary)
print(f"Wrote {summary_path}")